In [1]:
import gc                                # garbage collector, used to free RAM
import numpy as np                       # NumPy: numerical arrays
import pandas as pd                      # pandas: tables
import scanpy as sc                      # Scanpy: single-cell analysis
import anndata as ad                     # anndata: the data structure behind Scanpy (needed for concat)
import matplotlib.pyplot as plt          # matplotlib: plotting
from pathlib import Path                 # Path: file and folder paths

project_dir = Path.cwd().parent          # notebook lives in Notebook/, so parent = project root
proc_dir   = project_dir / "data" / "processed"
fig_dir    = project_dir / "Figure"
result_dir = project_dir / "Result"
fig_dir.mkdir(parents=True, exist_ok=True)

labels = ["D7", "D14", "D21"]
for s in labels:
    assert (proc_dir / f"{s}_qc.h5ad").exists(), f"Missing {s}_qc.h5ad (notebook 04)"
assert (result_dir / "scrublet_scores.csv").exists(), "Missing scrublet_scores.csv (notebook 05)"
print("Setup OK")

Setup OK


In [2]:
adatas = {}                                                # sample label -> AnnData object
for s in labels:
    a = sc.read_h5ad(proc_dir / f"{s}_qc.h5ad")            # QC-filtered cells of one sample
    print(s, a.shape)
    adatas[s] = a

# Stack the samples cell-wise.
#  label="sample"      -> adds an obs column "sample" with the dictionary keys (D7 / D14 / D21)
#  index_unique="_"    -> renames cells to <barcode>_<sample> so names stay unique
#  join="inner"        -> keep only genes present in all samples (all 36,601 should be kept)
#  merge="same"        -> keep gene-level columns (e.g. gene_id, mt) that are identical across samples
adata = ad.concat(adatas, label="sample", index_unique="_", join="inner", merge="same")
del adatas                                                 # free the per-sample copies
gc.collect()

# Attach Scrublet scores from notebook 05, matching on the same <barcode>_<sample> naming
scores = pd.read_csv(result_dir / "scrublet_scores.csv", index_col=0).rename_axis("barcode").reset_index()
scores.index = scores["barcode"] + "_" + scores["sample"]
assert adata.obs_names.isin(scores.index).all(), "Some cells have no Scrublet score"
adata.obs["doublet_score"]    = scores.loc[adata.obs_names, "doublet_score"].to_numpy()
adata.obs["scrublet_called"]  = scores.loc[adata.obs_names, "predicted_doublet"].to_numpy()

# Visualisation aid only (NOT a doublet call): flag the top 3% of scores within each sample
adata.obs["high_doublet_score"] = adata.obs.groupby("sample", observed=True)["doublet_score"].transform(
    lambda x: x >= x.quantile(0.97))

print(adata)
print(adata.obs["sample"].value_counts())

D7 (4540, 36601)
D14 (3246, 36601)
D21 (5454, 36601)
AnnData object with n_obs × n_vars = 13240 × 36601
    obs: 'total_counts', 'timepoint', 'n_genes_by_counts', 'total_counts_mt', 'pct_counts_mt', 'sample', 'doublet_score', 'scrublet_called', 'high_doublet_score'
    var: 'gene_id', 'mt'
sample
D21    5454
D7     4540
D14    3246
Name: count, dtype: int64


In [3]:
adata.layers["counts"] = adata.X.copy()      # keep the raw UMI counts (needed later, e.g. for differential expression)

# Scale every cell to 10,000 total counts so cells with different depth become comparable
sc.pp.normalize_total(adata, target_sum=1e4)

# log(1 + x): compresses the huge range of expression values so highly expressed genes do not dominate
sc.pp.log1p(adata)

In [4]:
# Select the 3,000 most variable genes. Variability is computed within each sample (batch_key),
# then the per-sample results are combined. "seurat" flavor works on the log-normalised values.
sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat", batch_key="sample")

n_hvg = int(adata.var["highly_variable"].sum())          # number of genes selected
n_all3 = int((adata.var["highly_variable_nbatches"] == 3).sum())   # genes variable in all 3 samples
print(f"HVGs selected: {n_hvg}  | variable in all 3 samples: {n_all3}")

HVGs selected: 3000  | variable in all 3 samples: 375


In [5]:
# Sanity check: undoing log1p on 5 cells should give a total close to 10,000 each
print(np.asarray(adata.X[:5].expm1().sum(axis=1)).ravel())

adata.write_h5ad(proc_dir / "merged_normalized.h5ad", compression="gzip")   # gzip makes the file smaller
print("Saved:", (proc_dir / "merged_normalized.h5ad").stat().st_size / 1024**2, "MB")


[ 9999.999 10000.    10000.001 10000.    10000.001]
Saved: 303.5686492919922 MB


In [6]:
# For each selected HVG: in how many of the 3 samples was it also called variable?
counts = adata.var.loc[adata.var["highly_variable"], "highly_variable_nbatches"].value_counts().sort_index()
print(counts)

highly_variable_nbatches
1    1301
2    1324
3     375
Name: count, dtype: int64
